# Teks — EDA untuk presentasi ("what we see"): semua grafik + angka tersimpan ke file
**Kapan dipakai:** begitu data lomba dibaca, sebelum/sejajar dengan `baseline_oof`. Saat membuat
PPT data lomba tidak bisa dibuka lagi, jadi notebook ini menyimpan SEMUA yang dibutuhkan slide:

- `NN_nama.png` (200 dpi, kanvas 13 x 5,4 in = pas di slide 16:9) dan `NN_nama.csv` (angka di
  balik grafik, nama sama: bisa digambar ulang di laptop tanpa data).
- `fakta.md`: semua temuan dalam kalimat + tabel kecil. Tempel ke AI online saat menyusun slide
  "what we see"; isinya hanya angka hasil hitungan, bukan teks mentah.
- `06_duplikat_contoh.csv` dan `09_contoh_teks.csv` berisi potongan teks ASLI (bukan agregat): cek
  aturan panitia sebelum membawanya keluar dari PC lomba.

**Isi:** 0 kolom → 1 kelas → 2 panjang teks → 3 kata khas per kelas (weighted log-odds, Monroe
2008) → 3b word cloud per kelas →
4 ciri teks (URL, emoji, mention, mojibake, ...) → 5 kosakata baru di test (OOV) →
6 kembar & hampir kembar train↔test (+ seberapa bisa dipercaya "salin label tetangga") →
7 adversarial validation + kata pembeda → 8 waktu (bila ada kolom tanggal) → 9 contoh teks →
fakta.md.

**Saat lomba yang diubah:** sel Konfigurasi (`# GANTI saat lomba`), lalu Run All. Target angka
(regresi) otomatis dibagi 4 kelompok kuantil. Multilabel: isi `TARGET` dengan satu kolom label.
Waktu demo (SmSA 5 rb baris): ± 1 menit di CPU.

In [ ]:
import os
import re
import textwrap
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.colors import ListedColormap
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors
from wordcloud import WordCloud

## Konfigurasi

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/eda_presentasi"         # semua file untuk PPT ada di sini

TEXT_COLS = ["text"]           # GANTI saat lomba: lebih dari satu kolom = digabung dengan spasi
TARGET = "label"               # GANTI saat lomba
DATE_COL = None                # GANTI: kolom tanggal (bagian 8), None = dilewati
MAX_CLASSES = 12               # kelas lebih banyak: kelas kecil digabung jadi "lainnya" di grafik
LANGUAGE = "id"                # "id" = stopword Sastrawi, "en" = stopword Inggris (sklearn)
TOKENIZER_PATH = None          # None = panjang dalam kata
# TOKENIZER_PATH = "../../../pretrained/indobert"   # panjang dalam token model (cek pemotongan)
LONG_LIMIT = 256               # batas "teks panjang" dalam satuan di atas (token: isi max_length)
NGRAM = (1, 1)                 # (1, 2) = kata khas termasuk frasa dua kata
MIN_DF = 5                     # kata khas: kata harus muncul di >= MIN_DF teks
N_TOP_WORDS = 10
NEAR_LEVEL = 0.9               # kemiripan cosine char n-gram untuk "hampir kembar"
NEAR_CHECK = 5000              # maksimum baris yang dicek hampir kembar (lambat bila besar)
N_EXAMPLES = 2                 # contoh teks per kelas di kartu
SEED = 42

## Gaya figur (aturan figur PPT repo)
Kanvas 13 x 5,4 in, huruf besar, grid putus-putus tipis di belakang data, tanpa judul di dalam
gambar (judul ditulis di slide). Font Be Vietnam Pro bila terpasang, kalau tidak Segoe UI/DejaVu.

In [ ]:
FIGURE_SIZE = (13, 5.4)
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9", "#7F7F7F", "#F0E442"]
installed = set()
for font in font_manager.fontManager.ttflist:
    installed.add(font.name)
families = []
for name in ["Be Vietnam Pro", "Segoe UI", "DejaVu Sans"]:
    if name in installed:
        families.append(name)
plt.rcParams.update({
    "font.family": families, "font.size": 15, "axes.titlesize": 16, "axes.labelsize": 15,
    "xtick.labelsize": 13, "ytick.labelsize": 13, "legend.fontsize": 13, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.grid": True, "grid.linestyle": "--", "grid.linewidth": 0.6, "grid.color": "#d9d9d9",
    "figure.dpi": 70, "savefig.dpi": 200, "savefig.bbox": "tight", "figure.facecolor": "white",
})
print("font:", families[0])

## Alat simpan
`save_figure` menyimpan PNG + CSV dengan nama sama dan mencatatnya untuk `fakta.md`.
`add_fact` menyimpan satu kalimat temuan per bagian. Angka ditulis gaya Indonesia (koma desimal).

In [ ]:
facts = {}
saved = []


def add_fact(section, text):
    facts.setdefault(section, []).append(text)
    print("-", text)


def save_figure(figure, name, table):
    figure.savefig(f"{OUTPUT_FOLDER}/{name}.png")
    table.to_csv(f"{OUTPUT_FOLDER}/{name}.csv", index=False)
    saved.append((name, table))
    plt.show()


def pct(value, digits=1):
    return f"{100 * value:.{digits}f}%".replace(".", ",")


def thousands(number):
    return f"{int(number):,}".replace(",", ".")


def class_color(position):
    return PALETTE[position % len(PALETTE)]


print("alat simpan siap:", OUTPUT_FOLDER)

## Baca data
Kolom teks digabung jadi satu. `label` = target sebagai teks; target angka dengan > 20 nilai unik
(regresi) dibagi 4 kelompok kuantil supaya bagian per kelas tetap jalan.

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)
# train = pd.read_json(TRAIN_CSV, lines=True)     # alt .jsonl (test juga)
# train = pd.read_excel(TRAIN_CSV)                # alt .xlsx
# train = pd.read_parquet(TRAIN_CSV)              # alt .parquet


def join_text(table):
    joined = table[TEXT_COLS[0]].fillna("").astype(str)
    for column in TEXT_COLS[1:]:
        joined = joined + " " + table[column].fillna("").astype(str)
    return joined.str.strip()


train_text = join_text(train)
test_text = join_text(test)
label = train[TARGET].astype(str)
classes = []
for value in sorted(train[TARGET].dropna().unique()):
    classes.append(str(value))
if pd.api.types.is_numeric_dtype(train[TARGET]) and train[TARGET].nunique() > 20:
    groups = pd.qcut(train[TARGET], 4, duplicates="drop")
    label = groups.astype(str)
    classes = list(groups.cat.categories.astype(str))
print("train:", train.shape, "| test:", test.shape, "| kelas:", classes[:10])

In [ ]:
if len(classes) > MAX_CLASSES:
    kept = list(label.value_counts().index[:MAX_CLASSES - 1])
    add_fact("Data", f"{len(classes)} kelas: grafik memakai {MAX_CLASSES - 1} kelas terbanyak, "
                     "sisanya digabung jadi 'lainnya'")
    label = label.where(label.isin(kept), "lainnya")
    classes = kept + ["lainnya"]
print("kelas di grafik:", len(classes))

## 0. Kolom dan ukuran data

In [ ]:
column_table = pd.DataFrame({"kolom": train.columns,
                             "tipe": train.dtypes.astype(str).to_numpy(),
                             "kosong_train": train.isna().sum(),
                             "unik_train": train.nunique()}, index=None)
column_table["ada_di_test"] = column_table["kolom"].isin(test.columns)
column_table.to_csv(f"{OUTPUT_FOLDER}/00_kolom.csv", index=False)
saved.append(("00_kolom", column_table))
other_columns = []
for column in train.columns:
    if column not in TEXT_COLS and column != TARGET:
        other_columns.append(column)
add_fact("Data", f"train {thousands(len(train))} baris x {train.shape[1]} kolom, "
                 f"test {thousands(len(test))} baris x {test.shape[1]} kolom")
add_fact("Data", f"kolom teks dipakai: {TEXT_COLS} | target {TARGET} ({len(classes)} kelas)")
add_fact("Data", f"kolom lain (tidak dipakai di EDA ini): {other_columns}")
add_fact("Data", f"teks kosong: train {(train_text == '').sum()}, test {(test_text == '').sum()} | "
                 f"label kosong di train: {train[TARGET].isna().sum()}")
print(column_table.to_string(index=False))

## 1. Sebaran kelas

In [ ]:
counts = label.value_counts().reindex(classes)
share = counts / counts.sum()
figure, ax = plt.subplots(figsize=FIGURE_SIZE)
colors = []
for position in range(len(classes)):
    colors.append(class_color(position))
ax.barh(classes, counts.to_numpy(), color=colors, height=0.6)
for i in range(len(classes)):
    ax.text(counts.iloc[i], i, f"  {thousands(counts.iloc[i])} ({pct(share.iloc[i])})", va="center")
ax.invert_yaxis()
ax.set_xlim(0, counts.max() * 1.3)
ax.set_xlabel("jumlah baris train")
ax.grid(axis="y", visible=False)
class_table = pd.DataFrame({"kelas": classes, "jumlah": counts.to_numpy(),
                            "porsi": share.round(4).to_numpy()})
save_figure(figure, "01_kelas", class_table)
ratio = counts.max() / max(counts.min(), 1)
add_fact("Label", f"kelas terbanyak {counts.idxmax()} ({pct(share.max())}), tersedikit "
                  f"{counts.idxmin()} ({pct(share.min())}), rasio {ratio:.1f}x")

## 2. Panjang teks per kelas dan train vs test
Satuan kata (cepat) atau token model (`TOKENIZER_PATH`): porsi > `LONG_LIMIT` token = porsi teks
yang terpotong bila `max_length = LONG_LIMIT`.

In [ ]:
unit = "kata"
train_length = train_text.str.split().str.len()
test_length = test_text.str.split().str.len()


def count_tokens(texts):
    encoded = tokenizer(texts.tolist(), add_special_tokens=True)["input_ids"]
    lengths = []
    for ids in encoded:
        lengths.append(len(ids))
    return pd.Series(lengths, index=texts.index)


if TOKENIZER_PATH is not None:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, local_files_only=True)
    unit = "token"
    train_length = count_tokens(train_text)
    test_length = count_tokens(test_text)
print("satuan panjang:", unit)

In [ ]:
limit = max(train_length.quantile(0.99), test_length.quantile(0.99), 1)
figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE, gridspec_kw={"width_ratios": [1.3, 1]})
groups = []
for name in classes:
    groups.append(train_length[label == name].clip(upper=limit))
boxes = axes[0].boxplot(groups, orientation="horizontal", showfliers=False, patch_artist=True,
                        medianprops={"color": "#1a1a1a", "linewidth": 2})
for position, box in enumerate(boxes["boxes"]):
    box.set_facecolor(class_color(position))
    box.set_alpha(0.7)
axes[0].set_yticks(range(1, len(classes) + 1), classes)
axes[0].invert_yaxis()
axes[0].set_xlabel(f"{unit} per teks (train)")
bins = np.linspace(0, limit, 40)
axes[1].hist(train_length.clip(upper=limit), bins=bins, density=True, alpha=0.5,
             color=PALETTE[0], label="train")
axes[1].hist(test_length.clip(upper=limit), bins=bins, density=True, histtype="step",
             linewidth=2.5, color=PALETTE[3], label="test")
if LONG_LIMIT < limit:
    axes[1].axvline(LONG_LIMIT, color="#555555", linestyle=":", linewidth=1.5)
    axes[1].text(LONG_LIMIT, axes[1].get_ylim()[1] * 0.95, f" {LONG_LIMIT}", color="#555555")
axes[1].set_xlabel(f"{unit} per teks (dipotong di p99 = {limit:.0f})")
axes[1].set_yticks([])
axes[1].legend()

In [ ]:
def length_row(name, values):
    return {"kelompok": name, "n": len(values), "median": values.median(),
            "p90": values.quantile(0.9), "p95": values.quantile(0.95), "maks": values.max(),
            f"porsi_lebih_{LONG_LIMIT}": round((values > LONG_LIMIT).mean(), 4)}


length_rows = []
for name in classes:
    length_rows.append(length_row(name, train_length[label == name]))
length_rows.append(length_row("train", train_length))
length_rows.append(length_row("test", test_length))
length_table = pd.DataFrame(length_rows)
save_figure(figure, "02_panjang", length_table)
longest = length_table.iloc[:len(classes)].sort_values("median").iloc[-1]
add_fact("Panjang", f"median {unit}: train {train_length.median():.0f}, test "
                    f"{test_length.median():.0f} | p95 train {train_length.quantile(0.95):.0f}")
add_fact("Panjang", f"teks > {LONG_LIMIT} {unit}: train {pct((train_length > LONG_LIMIT).mean())}, "
                    f"test {pct((test_length > LONG_LIMIT).mean())}")
add_fact("Panjang", f"kelas dengan teks terpanjang (median): {longest['kelompok']} "
                    f"({longest['median']:.0f} {unit})")

## 3. Kata khas per kelas (weighted log-odds, Monroe dkk. 2008)
Lebih jujur dari frekuensi (kata umum tidak menang) dan dari chi² (kata langka tidak meledak):
selisih log-odds kelas vs sisanya, prior Dirichlet = frekuensi korpus, dibagi simpangannya (z).
z > 2 kira-kira bermakna. Kata negasi sengaja TIDAK dibuang dari stopword.

In [ ]:
NEGATIONS = {"tidak", "bukan", "jangan", "belum", "kurang", "tak", "gak", "nggak", "not", "no"}
stop_words = "english"
if LANGUAGE == "id":
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    stop_words = sorted(set(StopWordRemoverFactory().get_stop_words()) - NEGATIONS)


def log_odds_z(counts, in_class):
    prior = np.asarray(counts.sum(axis=0)).ravel() + 0.01
    own = np.asarray(counts[in_class].sum(axis=0)).ravel()
    rest = np.asarray(counts[~in_class].sum(axis=0)).ravel()
    own_odds = np.log((own + prior) / (own.sum() + prior.sum() - own - prior))
    rest_odds = np.log((rest + prior) / (rest.sum() + prior.sum() - rest - prior))
    return (own_odds - rest_odds) / np.sqrt(1 / (own + prior) + 1 / (rest + prior))


word_counter = CountVectorizer(token_pattern=r"(?u)\b[^\W\d_]{2,}\b", stop_words=stop_words,
                               ngram_range=NGRAM, min_df=MIN_DF)
word_counts = word_counter.fit_transform(train_text)
vocabulary = word_counter.get_feature_names_out()
word_rows = []
for name in classes:
    z_values = log_odds_z(word_counts, (label == name).to_numpy())
    for index in np.argsort(-z_values)[:20]:
        word_rows.append({"kelas": name, "kata": vocabulary[index], "z": round(z_values[index], 2)})
words_table = pd.DataFrame(word_rows)
print("kosakata:", len(vocabulary), "kata")

In [ ]:
shown = classes[:6]
figure, axes = plt.subplots(1, len(shown), figsize=FIGURE_SIZE, squeeze=False)
for position, name in enumerate(shown):
    part = words_table[words_table["kelas"] == name].head(N_TOP_WORDS)
    ax = axes[0, position]
    ax.barh(part["kata"], part["z"], color=class_color(position), height=0.65)
    ax.invert_yaxis()
    ax.set_title(name)
    ax.set_xlabel("z log-odds")
    ax.grid(axis="y", visible=False)
figure.tight_layout()
save_figure(figure, "03_kata_khas", words_table)
for name in classes:
    part = words_table[words_table["kelas"] == name].head(N_TOP_WORDS)
    add_fact("Kata khas", f"{name}: {', '.join(part['kata'])}")
if len(classes) > len(shown):
    add_fact("Kata khas", f"grafik hanya {len(shown)} kelas pertama, semua kelas ada di CSV")

## 3b. Word cloud per kelas (untuk slide pembuka)
Ukuran kata = skor z log-odds bagian 3 (kata KHAS kelas itu), bukan frekuensi mentah, jadi kata
umum ("yang", "nya") tidak memenuhi gambar. Hanya kata dengan z > 0. Untuk analisis pakai bar
bagian 3; word cloud cuma ilustrasi. Alternatif frekuensi biasa: lihat komentar di sel.

In [ ]:
shown = classes[:6]
columns = min(len(shown), 3)
rows = int(np.ceil(len(shown) / columns))
figure, axes = plt.subplots(rows, columns, figsize=(13, 3.6 * rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
    ax.grid(False)
cloud_rows = []
for position, name in enumerate(shown):
    z_values = log_odds_z(word_counts, (label == name).to_numpy())
    # alternatif frekuensi: np.asarray(word_counts[(label == name).to_numpy()].sum(axis=0)).ravel()
    weights = {}
    for index in np.argsort(-z_values)[:80]:
        if z_values[index] > 0:
            weights[vocabulary[index]] = float(z_values[index])
    if len(weights) == 0:
        continue
    color = class_color(position)
    cloud = WordCloud(width=800, height=450, background_color="white", random_state=SEED,
                      prefer_horizontal=0.95, colormap=ListedColormap([color]))
    ax = axes.ravel()[position]
    ax.imshow(cloud.generate_from_frequencies(weights), interpolation="bilinear")
    ax.set_title(name, color=color, fontweight="bold")
    for word, weight in list(weights.items())[:30]:
        cloud_rows.append({"kelas": name, "kata": word, "z": round(weight, 2)})
figure.tight_layout()
save_figure(figure, "03b_wordcloud", pd.DataFrame(cloud_rows))

## 4. Ciri teks per kelas (porsi teks yang memiliki ciri)
Baris = ciri, kolom = kelas + train + test. Ciri yang beda jauh antar kelas = fitur/aturan
berguna; beda jauh train vs test = pembersihan harus sama di keduanya. Teks dibaca MENTAH.

In [ ]:
PATTERNS = {
    "URL": r"https?://|www\.",
    "mention @": r"@\w+",
    "hashtag #": r"#\w+",
    "emoji": "[\U0001F300-\U0001FAFF☀-➿]",
    "angka": r"\d",
    "tanda seru !": r"!",
    "tanda tanya ?": r"\?",
    "huruf berulang (3+)": r"([a-zA-Z])\1{2,}",
    "KATA KAPITAL": r"\b[A-Z]{3,}\b",
    "tag / entitas HTML": r"<[a-zA-Z/][^>]*>|&[a-z]+\x3b|&#\d+\x3b",   # \x3b = titik koma
    "mojibake (Ã, â€, ‚Äú)": "Ã.|â€|‚Ä|Â",
    "karakter non-ASCII": r"[^\x00-\x7F]",
}
trait_table = pd.DataFrame(index=list(PATTERNS))
for trait, pattern in PATTERNS.items():
    train_has = train_text.str.count(pattern) > 0
    for name in classes:
        trait_table.loc[trait, name] = train_has[label == name].mean()
    trait_table.loc[trait, "train"] = train_has.mean()
    trait_table.loc[trait, "test"] = (test_text.str.count(pattern) > 0).mean()
print(trait_table.round(3).to_string())
absent = trait_table.index[trait_table.max(axis=1) == 0]
if len(absent) < len(trait_table):
    add_fact("Ciri teks", f"tidak ada sama sekali (train dan test): {', '.join(absent)}")
    trait_table = trait_table.drop(index=absent)   # grafik hanya ciri yang muncul

In [ ]:
figure, ax = plt.subplots(figsize=FIGURE_SIZE)
values = trait_table.to_numpy(dtype=float)
ax.imshow(values, cmap="Blues", aspect="auto", vmin=0, vmax=max(values.max(), 0.01))
for row in range(values.shape[0]):
    for col in range(values.shape[1]):
        color = "white"
        if values[row, col] < values.max() * 0.6:
            color = "#1a1a1a"
        ax.text(col, row, pct(values[row, col]), ha="center", va="center", fontsize=12, color=color)
ax.set_xticks(range(values.shape[1]), trait_table.columns)
ax.set_yticks(range(values.shape[0]), trait_table.index)
ax.axvline(len(classes) - 0.5, color="white", linewidth=4)
ax.grid(False)
ax.tick_params(length=0)
trait_csv = trait_table.round(4).reset_index(names="ciri")
save_figure(figure, "04_ciri_teks", trait_csv)
spread = (trait_table[classes].max(axis=1) - trait_table[classes].min(axis=1)).sort_values()
for trait in spread.index[::-1][:3]:
    row = trait_table.loc[trait, classes]
    add_fact("Ciri teks", f"{trait}: {row.idxmax()} {pct(row.max())} vs "
                          f"{row.idxmin()} {pct(row.min())}")
gap = (trait_table["train"] - trait_table["test"]).abs()
for trait in gap[gap > 0.05].index:
    add_fact("Ciri teks", f"{trait} beda train vs test: {pct(trait_table.loc[trait, 'train'])} vs "
                          f"{pct(trait_table.loc[trait, 'test'])}")

## 5. Kosakata baru di test (OOV)
Kata test yang tidak pernah muncul di train. OOV tinggi → model kata (TF-IDF kata) lemah, pilih
char n-gram / subword (transformer) / TAPT; kata baru teratas sering = topik atau nama baru.

In [ ]:
letters = r"(?u)\b[^\W\d_]{2,}\b"
train_vocabulary = set(CountVectorizer(token_pattern=letters).fit(train_text).vocabulary_)
test_counter = CountVectorizer(token_pattern=letters)
test_matrix = test_counter.fit_transform(test_text)
test_words = test_counter.get_feature_names_out()
test_frequency = np.asarray(test_matrix.sum(axis=0)).ravel()
is_new = np.zeros(len(test_words), dtype=bool)
for index, word in enumerate(test_words):
    is_new[index] = word not in train_vocabulary
new_type_share = is_new.mean()
new_token_share = test_frequency[is_new].sum() / max(test_frequency.sum(), 1)
text_has_new = np.asarray(test_matrix[:, is_new].sum(axis=1)).ravel() > 0
new_table = pd.DataFrame({"kata": test_words[is_new], "frekuensi_test": test_frequency[is_new]})
new_table = new_table.sort_values("frekuensi_test", ascending=False).head(100)
summary = {"jenis kata": new_type_share, "kemunculan kata": new_token_share,
           "teks yang memuat kata baru": text_has_new.mean()}
add_fact("Kosakata", f"kata test yang tidak ada di train: {pct(new_type_share)} jenis kata, "
                     f"{pct(new_token_share)} kemunculan, "
                     f"{pct(text_has_new.mean())} teks test memuatnya")
add_fact("Kosakata", f"kata baru test teratas: {', '.join(new_table['kata'].head(10))}")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE, gridspec_kw={"width_ratios": [1, 1.2]})
names = list(summary)
axes[0].barh(names, list(summary.values()), color=PALETTE[3], height=0.55)
for i in range(len(names)):
    axes[0].text(summary[names[i]], i, "  " + pct(summary[names[i]]), va="center")
axes[0].invert_yaxis()
axes[0].set_xlim(0, max(summary.values()) * 1.4 + 0.01)
axes[0].set_xticks([])
axes[0].set_xlabel("porsi test yang baru (tidak ada di train)")
axes[0].grid(axis="y", visible=False)
top_new = new_table.head(15)
axes[1].barh(top_new["kata"], top_new["frekuensi_test"], color=PALETTE[1], height=0.65)
axes[1].invert_yaxis()
axes[1].set_xlabel("kemunculan di test (kata baru teratas)")
axes[1].grid(axis="y", visible=False)
figure.tight_layout()
save_figure(figure, "05_kosakata_baru", new_table)

## 6. Kembar dan hampir kembar
Teks dinormalkan (huruf kecil, hanya huruf/angka) lalu dicek: kembar di train, kembar beda label
(label berisik), test yang persis ada di train (bisa "lookup": salin label train), dan hampir
kembar (cosine char n-gram ≥ `NEAR_LEVEL`). **Uji lookup:** di train, label tetangga terdekat
(bukan dirinya) dibanding label asli = seberapa bisa dipercaya menyalin label tetangga.

In [ ]:
def normalize(texts):
    lowered = texts.str.lower()
    return lowered.str.replace(r"[^\w]+", " ", regex=True).str.strip()


train_key = normalize(train_text)
test_key = normalize(test_text)
repeated = train_key.duplicated(keep=False)
labels_per_text = label.groupby(train_key).nunique()
conflict_texts = labels_per_text[labels_per_text > 1].index
conflict_rows = train_key.isin(conflict_texts)
test_seen = test_key.isin(set(train_key))
add_fact("Kembar", f"train: {thousands(repeated.sum())} baris berteks kembar "
                   f"({pct(repeated.mean())}), {len(conflict_texts)} teks kembar BEDA label "
                   f"({thousands(conflict_rows.sum())} baris)")
add_fact("Kembar", f"test yang persis ada di train: {thousands(test_seen.sum())} baris "
                   f"({pct(test_seen.mean())})")
if test_seen.sum() > 0:
    seen_keys = set(test_key[test_seen])
    unanimous = labels_per_text[labels_per_text.index.isin(seen_keys)] == 1
    add_fact("Kembar", f"dari teks test itu, label train-nya seragam: {pct(unanimous.mean())} "
                       "(seragam = kandidat lookup)")

In [ ]:
rng = np.random.default_rng(SEED)
char_vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                                  sublinear_tf=True, max_features=300000)
train_chars = char_vectorizer.fit_transform(train_key)
neighbors = NearestNeighbors(n_neighbors=2, metric="cosine").fit(train_chars)
test_rows = np.sort(rng.permutation(len(test_key))[:NEAR_CHECK])
distance, nearest = neighbors.kneighbors(char_vectorizer.transform(test_key.iloc[test_rows]))
test_similarity = 1 - distance[:, 0]
test_nearest = nearest[:, 0]
train_rows = np.sort(rng.permutation(len(train_key))[:NEAR_CHECK])
distance, nearest = neighbors.kneighbors(train_chars[train_rows])
self_first = nearest[:, 0] == train_rows
other = np.where(self_first, nearest[:, 1], nearest[:, 0])
other_similarity = 1 - np.where(self_first, distance[:, 1], distance[:, 0])
close = other_similarity >= NEAR_LEVEL
agree = label.to_numpy()[train_rows] == label.to_numpy()[other]
near_test = (test_similarity >= NEAR_LEVEL).mean()
add_fact("Kembar", f"test hampir kembar dengan train (cosine >= {NEAR_LEVEL}): {pct(near_test)} "
                   f"(dicek {thousands(len(test_rows))} baris)")
if close.sum() > 0:
    add_fact("Kembar", f"uji lookup di train: tetangga dengan cosine >= {NEAR_LEVEL} berlabel "
                       f"sama {pct(agree[close].mean())} dari {thousands(close.sum())} baris, "
                       f"semua tetangga {pct(agree.mean())}")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE, gridspec_kw={"width_ratios": [1.2, 1]})
axes[0].hist(test_similarity, bins=np.linspace(0, 1, 41), color=PALETTE[0], alpha=0.8)
axes[0].axvline(NEAR_LEVEL, color=PALETTE[3], linestyle="--", linewidth=2)
axes[0].text(NEAR_LEVEL, axes[0].get_ylim()[1] * 0.9, f" ≥ {NEAR_LEVEL}: {pct(near_test)}",
             color=PALETTE[3], ha="right")
axes[0].set_xlabel("kemiripan test ke teks train terdekat (cosine)")
axes[0].set_ylabel("jumlah teks test")
bars = {"train berteks kembar": repeated.mean(), "kembar beda label": conflict_rows.mean(),
        "test persis ada di train": test_seen.mean(),
        f"test hampir kembar (≥ {NEAR_LEVEL})": near_test}
names = list(bars)
axes[1].barh(names, list(bars.values()), color=PALETTE[2], height=0.55)
for i in range(len(names)):
    axes[1].text(bars[names[i]], i, "  " + pct(bars[names[i]]), va="center")
axes[1].invert_yaxis()
axes[1].set_xlim(0, max(bars.values()) * 1.5 + 0.01)
axes[1].set_xticks([])
axes[1].set_xlabel("porsi baris")
axes[1].grid(axis="y", visible=False)
figure.tight_layout()
duplicate_table = pd.DataFrame({"ukuran": names, "porsi": np.round(list(bars.values()), 4)})
save_figure(figure, "06_duplikat", duplicate_table)

In [ ]:
pairs = pd.DataFrame({"kemiripan": test_similarity.round(3),
                      "teks_test": test_text.iloc[test_rows].str.slice(0, 200).to_numpy(),
                      "teks_train": train_text.iloc[test_nearest].str.slice(0, 200).to_numpy(),
                      "label_train": label.iloc[test_nearest].to_numpy()})
pairs = pairs[pairs["kemiripan"] >= NEAR_LEVEL].sort_values("kemiripan").head(50)
pairs.to_csv(f"{OUTPUT_FOLDER}/06_duplikat_contoh.csv", index=False)
print("contoh pasangan (teks asli) ditulis:", len(pairs), "baris")
pairs.head()

## 7. Train vs test: adversarial validation
Model menebak "baris ini dari test?". AUC ≈ 0,5 = train dan test sebaran sama (split acak aman);
AUC tinggi = ada pembeda (waktu, sumber, topik) → validasi harus meniru test. Kata dengan
koefisien paling positif = khas test, paling negatif = khas train.

In [ ]:
adversarial_texts = pd.concat([train_text, test_text], ignore_index=True)
is_test = np.r_[np.zeros(len(train_text)), np.ones(len(test_text))]
adversarial_vectorizer = TfidfVectorizer(token_pattern=letters, min_df=3, sublinear_tf=True)
adversarial_matrix = adversarial_vectorizer.fit_transform(adversarial_texts)
adversarial_model = LogisticRegression(C=1, max_iter=2000, class_weight="balanced")
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
adversarial_proba = cross_val_predict(adversarial_model, adversarial_matrix, is_test, cv=folds,
                                      method="predict_proba")[:, 1]
adversarial_auc = roc_auc_score(is_test, adversarial_proba)
adversarial_model.fit(adversarial_matrix, is_test)
weights = adversarial_model.coef_[0]
words = adversarial_vectorizer.get_feature_names_out()
order = np.argsort(weights)
picked = np.r_[order[:10], order[-10:]]
adversarial_table = pd.DataFrame({"kata": words[picked], "koefisien": weights[picked].round(3)})
verdict = "sebaran sama, split acak aman"
if adversarial_auc > 0.75:
    verdict = "beda jelas, validasi harus meniru test (per waktu/sumber/kelompok)"
elif adversarial_auc > 0.6:
    verdict = "agak beda, cek kata pembeda"
add_fact("Train vs test", f"adversarial AUC {adversarial_auc:.3f}: {verdict}")
add_fact("Train vs test", f"kata khas test: {', '.join(words[order[-8:]][::-1])}")

In [ ]:
figure, ax = plt.subplots(figsize=FIGURE_SIZE)
colors = []
for value in adversarial_table["koefisien"]:
    colors.append(PALETTE[0])
    if value > 0:
        colors[-1] = PALETTE[3]
ax.barh(adversarial_table["kata"], adversarial_table["koefisien"], color=colors, height=0.65)
ax.axvline(0, color="#1a1a1a", linewidth=0.8)
ax.set_xlabel(f"koefisien (← khas train | khas test →)   AUC {adversarial_auc:.3f}")
ax.grid(axis="y", visible=False)
save_figure(figure, "07_train_vs_test", adversarial_table)

## 8. Waktu (hanya bila `DATE_COL` diisi)
Kiri: jumlah baris per bulan train vs test (test sesudah train = soal masa depan → split waktu).
Kanan: porsi tiap kelas per bulan (label bergeser dari waktu ke waktu?).

In [ ]:
if DATE_COL is not None:
    train_month = pd.to_datetime(train[DATE_COL], errors="coerce").dt.to_period("M")
    test_month = pd.to_datetime(test[DATE_COL], errors="coerce").dt.to_period("M")
    per_month = pd.DataFrame({"train": train_month.value_counts(),
                              "test": test_month.value_counts()})
    per_month = per_month.fillna(0).sort_index()
    share_by_month = pd.crosstab(train_month, label, normalize="index").reindex(columns=classes)
    figure, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE)
    month_names = per_month.index.astype(str)
    axes[0].plot(month_names, per_month["train"], marker="o", color=PALETTE[0], label="train")
    axes[0].plot(month_names, per_month["test"], marker="s", color=PALETTE[3], label="test")
    axes[0].set_ylabel("baris per bulan")
    axes[0].legend()
    for position, name in enumerate(classes):
        axes[1].plot(share_by_month.index.astype(str), share_by_month[name], marker="o",
                     color=class_color(position), label=name)
    axes[1].set_ylabel("porsi kelas (train)")
    axes[1].legend()
    for ax in axes:
        ax.tick_params(axis="x", rotation=45)
    figure.tight_layout()
    save_figure(figure, "08_waktu", per_month.reset_index(names="bulan").astype(str))
    add_fact("Waktu", f"train {train_month.min()} s/d {train_month.max()}, "
                      f"test {test_month.min()} s/d {test_month.max()}")
else:
    print("DATE_COL kosong: bagian waktu dilewati")

## 9. Contoh teks per kelas (kartu untuk slide)
Diambil acak dari teks berpanjang wajar (kuartil 1–3), dipotong 300 karakter. Isinya teks ASLI.

In [ ]:
lengths = train_text.str.len()
typical = lengths.between(lengths.quantile(0.25), lengths.quantile(0.75))
example_rows = []
for name in classes[:6]:
    pool = train_text[(label == name) & typical]
    if len(pool) == 0:
        pool = train_text[label == name]
    for index in rng.choice(pool.index, size=min(N_EXAMPLES, len(pool)), replace=False):
        example_rows.append({"kelas": name, "teks": train_text[index][:300]})
examples = pd.DataFrame(example_rows)
examples["kolom"] = examples.groupby("kelas").cumcount()
height = max(FIGURE_SIZE[1], 1.5 * len(classes[:6]))
figure, axes = plt.subplots(len(classes[:6]), N_EXAMPLES, figsize=(13, height), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for _, row in examples.iterrows():
    ax = axes[classes.index(row["kelas"]), row["kolom"]]
    wrapped = textwrap.fill(row["teks"], 70, max_lines=4, placeholder=" …")
    ax.text(0, 1, row["kelas"], fontsize=14, fontweight="bold", va="top",
            color=class_color(classes.index(row["kelas"])), transform=ax.transAxes)
    ax.text(0, 0.72, wrapped, fontsize=12, va="top", transform=ax.transAxes)
figure.tight_layout()
figure.savefig(f"{OUTPUT_FOLDER}/09_contoh_teks.png")
examples.to_csv(f"{OUTPUT_FOLDER}/09_contoh_teks.csv", index=False)
plt.show()

## fakta.md (tempel ke AI online saat membuat slide)
Semua kalimat temuan + tabel di balik tiap grafik (maks 30 baris per tabel). Hanya agregat.

In [ ]:
lines = ["# Fakta data (otomatis dari eda_presentasi)", "",
         f"Dibuat {time.strftime('%Y-%m-%d %H:%M')} dari `{TRAIN_CSV}` dan `{TEST_CSV}`.",
         "Untuk AI pembuat slide: semua angka di bawah dihitung dari data.",
         "Jangan menambah angka atau contoh yang tidak tertulis di sini.",
         "Nama file PNG/CSV di folder yang sama.", ""]
for section, items in facts.items():
    lines.append(f"## {section}")
    for item in items:
        lines.append(f"- {item}")
    lines.append("")
lines.append("## Tabel di balik grafik")
for name, table in saved:
    lines.append(f"### {name}.png / {name}.csv")
    lines.append("```")
    lines.append(table.head(30).to_string(index=False))
    lines.append("```")
with open(f"{OUTPUT_FOLDER}/fakta.md", "w", encoding="utf-8") as report:
    report.write("\n".join(lines))
print("ditulis:", f"{OUTPUT_FOLDER}/fakta.md")
print(sorted(os.listdir(OUTPUT_FOLDER)))